In [1]:
# ==========================================
# Causal-frame network divergence
#
# Independently extract causal frames from chunks_all.csv and compare directed
# cause-topic / relation / effect-topic networks. No semantic-coverage output is
# read by this method.
# ==========================================

import html
import io
import os

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
import re
import shutil
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from PIL import Image
from scipy.spatial.distance import jensenshannon
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 260)

RANDOM_STATE = 42
MIN_COUNTRY_ITEMS = 10
MIN_COUNTRY_SOURCES = 2
MIN_FRAME_QUALITY = 0.40
MAX_SENTENCE_WORDS = 120
MIN_SPAN_WORDS = 2
MAX_SPAN_WORDS = 55
BOOTSTRAP_ITERATIONS = 250

BLUE = "#1f77b4"
ORANGE = "#ff7f0e"
GAP_CMAP = LinearSegmentedColormap.from_list(
    "sentiment_to_policy_gap",
    [ORANGE, "#ffffff", BLUE],
)


def find_project_root() -> Path:
    """Find the repository root from the expected NMF and chunk paths."""
    env_root = os.environ.get("CAUSAL_NLP_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (
            (candidate / "progress" / "topic_modelling" / "nmf").exists()
            and (candidate / "csvs" / "chunked" / "chunks_all.csv").exists()
        ):
            return candidate

    for start in [Path.cwd(), *Path.cwd().parents]:
        if (
            (start / "progress" / "topic_modelling" / "nmf").exists()
            and (start / "csvs" / "chunked" / "chunks_all.csv").exists()
        ):
            return start.resolve()

    raise FileNotFoundError(
        "Could not find progress/topic_modelling/nmf and "
        "csvs/chunked/chunks_all.csv. Run from the repository root or set "
        "CAUSAL_NLP_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()
NMF_ROOT = PROJECT_ROOT / "progress" / "topic_modelling" / "nmf"
CSVS_CHUNKED = PROJECT_ROOT / "csvs" / "chunked"
CHUNKS_ALL = CSVS_CHUNKED / "chunks_all.csv"
METHOD_ROOT = Path(
    os.environ.get(
        "CAUSAL_NLP_OUTPUT_ROOT",
        PROJECT_ROOT / "progress" / "causal_nlp",
    )
).expanduser().resolve()
POLICY_OUTPUT = NMF_ROOT / "policy" / "output" / "global"
SENTIMENT_OUTPUT = NMF_ROOT / "sentiment" / "output"



# Shared causal-text cleaning layer
import sys

_SHARED_CANDIDATES = [
    Path(os.environ.get("CAUSAL_NLP_SHARED_MODULE_DIR", "")).expanduser(),
    PROJECT_ROOT / "progress" / "causal_nlp" / "shared",
    Path.cwd() / "shared",
    Path.cwd(),
]
SHARED_MODULE_DIR = next(
    (
        candidate.resolve()
        for candidate in _SHARED_CANDIDATES
        if str(candidate) and (candidate / "causal_text_cleaning.py").exists()
    ),
    None,
)
if SHARED_MODULE_DIR is None:
    raise FileNotFoundError(
        "causal_text_cleaning.py was not found. Place it in "
        "progress/causal_nlp/shared or set CAUSAL_NLP_SHARED_MODULE_DIR."
    )
if str(SHARED_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_MODULE_DIR))

from causal_text_cleaning import (
    CAUSAL_ARTIFACT_PHRASES,
    CAUSAL_ARTIFACT_TOKENS,
    CAUSAL_EMBEDDING_STOPWORDS,
    CLEANING_VERSION,
    build_context_windows,
    build_topic_description,
    contains_contact_or_link,
    content_token_ratio,
    infer_sentiment_country,
    is_source_residue,
    load_or_build_clean_sentence_inventory,
    normalize_source_text,
    sentence_units,
)

# Compatibility names used by the method-specific extraction code.
normalize_for_noise_check = normalize_source_text
split_sentences = sentence_units


def normalise_span(text: str) -> str:
    return re.sub(r"[^a-z0-9à-ÿ]+", " ", str(text).lower()).strip()


def normalise_minmax(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0.0)
    minimum = float(values.min())
    maximum = float(values.max())
    if maximum <= minimum:
        return pd.Series(np.zeros(len(values)), index=series.index)
    return (values - minimum) / (maximum - minimum)


def save_verified_png(fig, output_path: Path, dpi: int = 160) -> None:
    """Write a standard RGB PNG directly to the final path and verify it."""
    output_path = Path(output_path).expanduser().resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    raw_buffer = io.BytesIO()
    rgb_buffer = io.BytesIO()
    try:
        fig.savefig(
            raw_buffer,
            format="png",
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
            transparent=False,
        )
        plt.close(fig)
        raw_buffer.seek(0)
        with Image.open(raw_buffer) as image:
            image.load()
            image.convert("RGB").save(
                rgb_buffer,
                format="PNG",
                optimize=False,
                compress_level=6,
            )
        png_bytes = rgb_buffer.getvalue()
        if not png_bytes.startswith(b"\x89PNG\r\n\x1a\n") or len(png_bytes) <= 100:
            raise OSError("Invalid PNG bytes were generated.")
        output_path.unlink(missing_ok=True)
        output_path.write_bytes(png_bytes)
        with Image.open(output_path) as image:
            image.load()
            if image.format != "PNG" or image.mode != "RGB":
                raise OSError(f"Invalid final PNG: {output_path}")
            width, height = image.size
        print("Saved verified PNG:", output_path, f"size=({width}, {height})")
    finally:
        plt.close(fig)
        raw_buffer.close()
        rgb_buffer.close()


In [2]:
# ==========================================
# Step 1: Build or load the shared clean sentence inventory, then attach NMF topics
# ==========================================

OUTPUT_DIR = METHOD_ROOT / "output" / "frame_network"
IMG_DIR = METHOD_ROOT / "img" / "frame_network"
SHARED_OUTPUT_DIR = METHOD_ROOT / "output" / "shared"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)
SHARED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_SENTENCE_INVENTORY = SHARED_OUTPUT_DIR / "clean_sentence_inventory.csv"
CLEAN_SENTENCE_METADATA = SHARED_OUTPUT_DIR / "clean_sentence_inventory_metadata.json"
CLEAN_SENTENCE_SUMMARY = SHARED_OUTPUT_DIR / "clean_sentence_inventory_summary.csv"
CLEAN_SENTENCE_VALIDATION = SHARED_OUTPUT_DIR / "clean_sentence_inventory_validation.csv"

clean_sentence_inventory, shared_cleaning_metadata = (
    load_or_build_clean_sentence_inventory(
        CHUNKS_ALL,
        CLEAN_SENTENCE_INVENTORY,
        CLEAN_SENTENCE_METADATA,
        CLEAN_SENTENCE_SUMMARY,
        CLEAN_SENTENCE_VALIDATION,
    )
)
raw_chunks_all = build_context_windows(pd.read_csv(CHUNKS_ALL))

policy_topics = pd.read_csv(
    POLICY_OUTPUT / "policy_global_nmf_topic_info_with_labels_final.csv"
).sort_values("topic").reset_index(drop=True)
policy_assignments = pd.read_csv(
    POLICY_OUTPUT / "policy_global_nmf_documents_with_topic_labels_final.csv"
)[["chunk_id", "topic", "topic_confidence"]]
sentiment_assignments = pd.read_csv(
    SENTIMENT_OUTPUT / "sentiment_nmf_original_documents_with_topic_labels_final.csv"
)[["chunk_id", "topic", "topic_confidence"]]
synthetic_assignments = pd.read_csv(
    SENTIMENT_OUTPUT / "sentiment_nmf_synthetic_assignments_with_topic_labels_final.csv"
)[["chunk_id", "assigned_topic", "topic_confidence"]].rename(
    columns={"assigned_topic": "topic"}
)

for name, assignment in [
    ("policy", policy_assignments),
    ("sentiment", sentiment_assignments),
    ("synthetic", synthetic_assignments),
]:
    if assignment["chunk_id"].duplicated().any():
        raise ValueError(f"Duplicate frozen NMF chunk IDs found for {name}.")


def attach_clean_sentences(
    assignments: pd.DataFrame,
    corpus: str,
    source_type: str,
) -> pd.DataFrame:
    selected = clean_sentence_inventory[
        clean_sentence_inventory["corpus"].eq(corpus)
        & clean_sentence_inventory["source_type"].eq(source_type)
    ].merge(assignments, on="chunk_id", how="inner", validate="many_to_one")
    context_columns = raw_chunks_all[
        ["chunk_id", "chunk_text", "context_window"]
    ].drop_duplicates("chunk_id")
    selected = selected.merge(
        context_columns,
        on="chunk_id",
        how="left",
        validate="many_to_one",
    )
    selected["original_chunk_text"] = selected["chunk_text"]
    selected["chunk_text"] = selected["clean_sentence"]
    return selected


policy_chunks = attach_clean_sentences(policy_assignments, "policy", "original")
sentiment_chunks = attach_clean_sentences(
    sentiment_assignments, "sentiment", "original"
)
synthetic_chunks = attach_clean_sentences(
    synthetic_assignments, "sentiment", "synthetic"
)

policy_chunks["analysis_country"] = (
    policy_chunks["country"].fillna("other").astype(str).str.lower()
)
sentiment_chunks["analysis_country"] = sentiment_chunks.apply(
    infer_sentiment_country,
    axis=1,
)
synthetic_chunks["analysis_country"] = "synthetic"

linkage = pd.DataFrame([
    {
        "corpus": "policy",
        "frozen_nmf_chunks": len(policy_assignments),
        "raw_linked_chunks": raw_chunks_all[
            raw_chunks_all["chunk_id"].isin(policy_assignments["chunk_id"])
        ]["chunk_id"].nunique(),
        "clean_linked_chunks": policy_chunks["chunk_id"].nunique(),
        "clean_sentences": len(policy_chunks),
    },
    {
        "corpus": "sentiment",
        "frozen_nmf_chunks": len(sentiment_assignments),
        "raw_linked_chunks": raw_chunks_all[
            raw_chunks_all["chunk_id"].isin(sentiment_assignments["chunk_id"])
        ]["chunk_id"].nunique(),
        "clean_linked_chunks": sentiment_chunks["chunk_id"].nunique(),
        "clean_sentences": len(sentiment_chunks),
    },
    {
        "corpus": "synthetic_sentiment",
        "frozen_nmf_chunks": len(synthetic_assignments),
        "raw_linked_chunks": raw_chunks_all[
            raw_chunks_all["chunk_id"].isin(synthetic_assignments["chunk_id"])
        ]["chunk_id"].nunique(),
        "clean_linked_chunks": synthetic_chunks["chunk_id"].nunique(),
        "clean_sentences": len(synthetic_chunks),
    },
])
linkage["raw_linkage_rate"] = (
    linkage["raw_linked_chunks"]
    / linkage["frozen_nmf_chunks"].clip(lower=1)
)
linkage.to_csv(OUTPUT_DIR / "original_chunk_linkage.csv", index=False)
if linkage["raw_linkage_rate"].lt(0.95).any():
    raise ValueError("Raw original-chunk linkage fell below 95%.")

print("Shared cleaning version:", CLEANING_VERSION)
print("Shared inventory hash:", shared_cleaning_metadata["inventory_sha256"])
print(linkage.to_string(index=False))


Shared cleaning version: shared-causal-text
Shared inventory hash: 59f25d414f8672b5fd0423a5c35ea1452e3d3d051a4d045d2c378c142c40f96a
             corpus  frozen_nmf_chunks  raw_linked_chunks  clean_linked_chunks  clean_sentences  raw_linkage_rate
             policy               1888               1888                 1819            14944               1.0
          sentiment                455                455                  450             3578               1.0
synthetic_sentiment                202                202                  202             2040               1.0


In [3]:
# ==========================================
# Step 2: Independently mine quality-scored causal frames
# ==========================================


def cue_regex(cues: str) -> re.Pattern:
    return re.compile(rf"(?<!\w)(?:{cues})(?!\w)", flags=re.IGNORECASE)


MODAL = r"(?:to|can|could|may|might|must|should|will|would|shall)"

FRAME_CUES = [
    (
        cue_regex(
            r"because of|because|due to|owing to|as a result of|resulting from|"
            r"caused by|driven by|triggered by"
        ),
        "causes_or_increases",
        "reverse",
        1.00,
    ),
    (
        cue_regex(
            r"may lead to|can lead to|could lead to|is likely to lead to|"
            r"lead(?:s|ing)? to|gives? rise to|result(?:s|ed|ing)? in|"
            r"bring(?:s|ing)? about|brought about|caus(?:e|es|ed|ing)|"
            r"contribut(?:e|es|ed|ing) to|drive|drives|driving|"
            r"produc(?:e|es|ed|ing)|trigger(?:s|ed|ing)?|"
            rf"(?:{MODAL}\s+generate|generates|generating)|"
            rf"(?:{MODAL}\s+create|creates|creating)|"
            r"exacerbat(?:e|es|ed|ing)|worsen(?:s|ed|ing)?|"
            r"intensif(?:y|ies|ied|ying)|accelerat(?:e|es|ed|ing)|"
            r"increas(?:e|es|ed|ing)|rais(?:e|es|ed|ing)|"
            r"amplif(?:y|ies|ied|ying)|heighten(?:s|ed|ing)?"
        ),
        "causes_or_increases",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            rf"(?:{MODAL}\s+reduce|reduces|reducing)|"
            rf"(?:{MODAL}\s+prevent|prevents|preventing)|"
            rf"(?:{MODAL}\s+limit|limits|limiting)|"
            rf"(?:{MODAL}\s+mitigate|mitigates|mitigating)|"
            rf"(?:{MODAL}\s+decrease|decreases|decreasing)|"
            rf"(?:{MODAL}\s+lower|lowers|lowering)|"
            r"minimi(?:s|z)(?:es|ing)|"
            r"alleviat(?:es|ing)|avoid(?:s|ing)?|curb(?:s|ing)?|"
            r"constrain(?:s|ing)?|counteract(?:s|ing)?|"
            r"protect(?:s|ing)? against|safeguard(?:s|ing)? against"
        ),
        "reduces_or_prevents",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            rf"(?:{MODAL}\s+support(?:\s+and\s+assist)?|supports|supporting)|"
            rf"(?:{MODAL}\s+help(?:\s+to|\s+with|\s+in)?|"
            r"helps(?:\s+to|\s+with|\s+in)?|"
            r"helped(?:\s+to|\s+with|\s+in)?|"
            r"helping(?:\s+to|\s+with|\s+in)?)|"
            rf"(?:{MODAL}\s+enable|enables|enabling)|"
            rf"(?:{MODAL}\s+facilitate|facilitates|facilitating)|"
            rf"(?:{MODAL}\s+allow(?:\s+for)?|allows(?:\s+for)?|allowing(?:\s+for)?)|"
            r"mak(?:e|es|ing) possible|"
            r"made possible|foster(?:s|ing)?|encourag(?:e|es|ing)|"
            r"promot(?:e|es|ing)|empower(?:s|ing)?|"
            r"assist(?:s|ing)?(?:\s+with|\s+in)?|"
            r"in order to|so as to|with the aim of|for the purpose of|"
            r"with a view to|designed to|intended to"
        ),
        "enables_or_supports",
        "forward",
        0.95,
    ),
    (
        cue_regex(r"enabled by|facilitated by|made possible by"),
        "enables_or_supports",
        "reverse",
        0.95,
    ),
    (
        cue_regex(
            r"is required for|are required for|was required for|were required for|"
            r"is necessary for|are necessary for|was necessary for|were necessary for"
        ),
        "requires_or_depends_on",
        "forward",
        0.95,
    ),
    (
        cue_regex(
            r"requir(?:e|es|ing)|depend(?:s|ing)? on|"
            r"relies? on|relying on|necessitat(?:e|es|ing)|"
            r"presuppos(?:e|es|ing)|is contingent on|is conditional on"
        ),
        "requires_or_depends_on",
        "reverse",
        0.95,
    ),
    (
        cue_regex(
            r"creat(?:e|es|ed|ing) a risk of|"
            r"increas(?:e|es|ed|ing) the risk of|"
            r"rais(?:e|es|ed|ing) the risk of|"
            r"pos(?:e|es|ing) a risk to|risk(?:ed|ing)|"
            r"threaten(?:s|ed|ing)?|undermin(?:e|es|ed|ing)|"
            r"harm(?:s|ed|ing)?|jeopardi(?:s|z)(?:e|es|ed|ing)|"
            r"endanger(?:s|ed|ing)?|compromis(?:e|es|ed|ing)|"
            r"weaken(?:s|ed|ing)?"
        ),
        "risks_or_threatens",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            r"is expected to improve|is intended to improve|"
            rf"(?:{MODAL}\s+improve|improves|improving)|"
            rf"(?:{MODAL}\s+enhance|enhances|enhancing)|"
            rf"(?:{MODAL}\s+strengthen|strengthens|strengthening)|"
            rf"(?:{MODAL}\s+advance|advances|advancing)|"
            r"boost(?:s|ing)?|optimi(?:s|z)(?:es|ing)"
        ),
        "expected_to_improve",
        "forward",
        0.95,
    ),
    (
        cue_regex(
            r"parce que|en raison de|à cause de|du fait de|résultant de|"
            r"causé par|causée par|provoqué par|provoquée par|"
            r"entraîné par|entraînée par|généré par|générée par"
        ),
        "causes_or_increases",
        "reverse",
        1.00,
    ),
    (
        cue_regex(
            r"peut conduire à|pourrait conduire à|est susceptible de conduire à|"
            r"conduit à|conduit au|conduit aux|donne lieu à|"
            r"entraîn(?:e|ent|ant)|caus(?:e|ent|ant)|"
            r"provoqu(?:e|ent|ant)|"
            r"engendr(?:e|ent|ant)|"
            r"contribu(?:e|ent|ant) à|"
            r"génèr(?:e|ent|ant)|déclench(?:e|ent|ant)|"
            r"accentu(?:e|ent|ant)|aggrav(?:e|ent|ant)|"
            r"accroît|accroissent|augment(?:e|ent|ant)|"
            r"amplifi(?:e|ent|ant)"
        ),
        "causes_or_increases",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            r"rédui(?:t|ts|te|tes|re|sent|sant)|prévien(?:t|nent)|"
            r"limit(?:e|ent|ant)|atténu(?:e|ent|ant)|"
            r"diminu(?:e|ent|ant)|abaiss(?:e|ent|ant)|"
            r"minimis(?:e|ent|ant)|évit(?:e|ent|ant)|"
            r"frein(?:e|ent|ant)|contraint|contraignent|"
            r"protèg(?:e|ent|ant) contre"
        ),
        "reduces_or_prevents",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            r"rend possible|rendent possible|permet(?:s|tent)? de|"
            r"permet(?:s|tent)?|soutien(?:t|nent)|"
            r"facilit(?:e|ent|ant)|"
            r"favoris(?:e|ent|ant)|"
            r"encourag(?:e|ent|ant)|"
            r"aid(?:e|ent|ant) à|donne les moyens de|"
            r"afin de|dans le but de|en vue de|de manière à|de façon à|"
            r"destiné à|destinée à|conçu pour|conçue pour|visant à"
        ),
        "enables_or_supports",
        "forward",
        0.95,
    ),
    (
        cue_regex(r"rendu possible par|rendue possible par"),
        "enables_or_supports",
        "reverse",
        0.95,
    ),
    (
        cue_regex(
            r"est nécessaire pour|sont nécessaires pour|"
            r"est requis pour|sont requis pour|est requise pour|sont requises pour"
        ),
        "requires_or_depends_on",
        "forward",
        0.95,
    ),
    (
        cue_regex(
            r"nécessit(?:e|es|ant)|dépend(?:s|ent)? de|"
            r"repos(?:e|es|ant) sur|exig(?:e|es|ant)|requiert|requièrent|"
            r"est tributaire de|"
            r"est conditionné par|est conditionnée par"
        ),
        "requires_or_depends_on",
        "reverse",
        0.95,
    ),
    (
        cue_regex(
            r"crée un risque de|créent un risque de|accroît le risque de|"
            r"augmente le risque de|présente un risque pour|"
            r"menac(?:e|es|é|ée|és|ées|ant)|risque de|"
            r"compromet|compromettent|nui(?:t|sent) à|met en danger|"
            r"mettent en danger|fragilis(?:e|es|é|ée|és|ées|ant)"
        ),
        "risks_or_threatens",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            r"devrait améliorer|vise à améliorer|"
            r"amélior(?:e|ent|ant)|"
            r"renforc(?:e|ent|ant)|"
            r"optimis(?:e|ent|ant)"
        ),
        "expected_to_improve",
        "forward",
        0.95,
    ),
]

NOUN_PREDECESSORS = {
    "a", "an", "the", "this", "that", "these", "those", "of", "for",
    "with", "without", "under", "over", "additional", "administrative",
    "continued", "digital", "educational", "existing", "financial", "human",
    "institutional", "ongoing", "practical", "professional", "social",
    "technical", "alongside", "through", "via", "including", "concerning",
    "regarding", "around",
}

SUPPORT_NOUN_FOLLOWERS = {
    "staff", "worker", "workers", "service", "services", "team", "teams",
    "function", "functions", "material", "materials", "resource", "resources",
    "network", "networks", "group", "groups", "structure", "structures",
    "mechanism", "mechanisms", "work", "personnel", "for", "of", "from",
    "to", "and", "or",
}


def sentence_units(text: str) -> list[str]:
    cleaned = normalize_source_text(text)
    cleaned = re.sub(r"\s+[•●▪◦]\s+", ". ", cleaned)
    cleaned = re.sub(
        r"\s+-\s+(?=[A-ZÀ-ÖØ-Ý][A-Za-zÀ-ÖØ-öø-ÿ ]{2,}:)",
        ". ",
        cleaned,
    )
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    cleaned = re.sub(
        r"([.!?])(['’”\"])?\s+(?=[A-ZÀ-ÖØ-Ý])",
        lambda match: match.group(1) + (match.group(2) or "") + "\n",
        cleaned,
    )
    return [
        unit.strip(" -•\t")
        for unit in re.split(r"\n|(?<=[.!?])\s+(?=\S)", cleaned)
        if unit.strip()
    ]


def trim_left(text: str, max_words: int = MAX_SPAN_WORDS) -> str:
    clause = re.split(r"[;:]", text)[-1]
    return " ".join(clause.strip(" ,;- ").split()[-max_words:])


def trim_right(text: str, max_words: int = MAX_SPAN_WORDS) -> str:
    clause = re.split(r"[;:]", text)[0]
    return " ".join(clause.strip(" ,;- ").split()[:max_words])


def clean_span(value: str) -> str:
    cleaned = normalize_source_text(value)
    cleaned = re.sub(
        r"^(?:chapter|section|article|figure|table|appendix|annex|"
        r"chapitre|partie|figure|tableau|annexe)\s+[A-Za-z0-9.:-]+\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"^(?:and|or|et|ou)\s+", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s+(?:to|de|à|pour)$", "", cleaned, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", cleaned).strip(" ,;:-")[:500]


def is_low_information_span(value: str) -> bool:
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9]+", str(value).lower())
    if len(tokens) < MIN_SPAN_WORDS or len(tokens) > MAX_SPAN_WORDS:
        return True
    if content_token_ratio(value) < 0.46:
        return True
    artifact_hits = sum(token in CAUSAL_ARTIFACT_TOKENS for token in tokens)
    return artifact_hits >= 3


def is_plausible_cue_usage(sentence: str, match: re.Match, relation_type: str) -> bool:
    cue = re.sub(r"\s+", " ", match.group(0).lower()).strip()
    before_tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        sentence[:match.start()].lower(),
    )
    after_tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        sentence[match.end():].lower(),
    )
    previous_word = before_tokens[-1] if before_tokens else ""
    next_word = after_tokens[0] if after_tokens else ""
    previous_character = sentence[match.start() - 1] if match.start() > 0 else ""
    next_character = sentence[match.end()] if match.end() < len(sentence) else ""

    # Hyphenated participles such as AI-amplified or AI-generated are
    # modifiers, not standalone causal predicates.
    if previous_character == "-" or next_character == "-":
        return False

    if cue in {"cause", "risk", "support", "need", "limit", "increase", "decrease"}:
        if previous_word in NOUN_PREDECESSORS:
            return False

    if cue == "cause" and previous_word in {"a", "the", "rather", "root", "main"}:
        return False
    if cue in {"support", "supporting"}:
        if previous_word in {
            "provide", "provides", "provided", "providing", "obtain", "obtains",
            "obtained", "obtaining", "receive", "receives", "received",
            "receiving", "funding", "scientific", "pillar", "competency",
            "section", "chapter", "objective", "action", "and", "or",
            "understanding",
        }:
            return False
        if next_word in SUPPORT_NOUN_FOLLOWERS:
            return False
    if cue in {"risk", "risks"}:
        if previous_word in {
            "a", "an", "the", "other", "potential", "greatest", "significant",
            "various", "several", "these", "those", "related", "associated",
        }:
            return False
        if next_word in {
            "is", "are", "was", "were", "that", "to", "across", "related",
            "associated", "identified", "include", "including",
        }:
            return False
    if cue == "raised" and next_word == "by":
        return False

    if cue in {
        "improved", "enhanced", "strengthened", "increased", "reduced",
        "limited", "generated", "created",
    } and previous_word in {
        "a", "an", "the", "and", "or", "new", "better", "more", "less",
        "with", "for", "of", "is", "are", "was", "were", "be", "been",
        "being", "remains", "remained",
    }:
        return False
    if cue.endswith("ing") and previous_word in {
        "in", "for", "of", "by", "through", "while", "after", "before",
        "during", "as", "including",
    }:
        return False
    if cue.startswith("contribut") and previous_word in {"why", "for", "by", "of"}:
        return False

    if cue in {"facilité", "facilités"}:
        return False

    if cue.startswith("aide à") and previous_word in {"l", "la", "une", "cette", "son"}:
        return False

    before_lowered = sentence[:match.start()].lower()
    if cue.endswith("ing") and (
        "types of" in before_lowered
        or re.search(r"\b(?:include|includes|included|including)\b", before_lowered)
    ):
        return False

    if next_word in {
        "and", "or", "but", "not", "et", "ou", "mais", "for", "of",
        "from", "with",
    }:
        return False
    return True


def span_quality(cause: str, effect: str, cue_weight: float) -> float:
    left = len(cause.split())
    right = len(effect.split())
    completeness = min(1.0, (left + right) / 28.0)
    balance = min(left, right) / max(left, right)
    content = min(content_token_ratio(cause), content_token_ratio(effect))
    length_penalty = 1.0 if max(left, right) <= 40 else 0.85
    return float(
        cue_weight
        * length_penalty
        * (0.48 * completeness + 0.30 * balance + 0.22 * content)
    )


def extract_frames(frame: pd.DataFrame, corpus_type: str) -> pd.DataFrame:
    candidates = []
    for _, document in frame.iterrows():
        for sentence_index, sentence in [(int(document["sentence_index"]), document["clean_sentence"])]:
            if is_source_residue(sentence):
                continue
            for cue_pattern, relation_type, orientation, cue_weight in FRAME_CUES:
                for match in cue_pattern.finditer(sentence):
                    if not is_plausible_cue_usage(sentence, match, relation_type):
                        continue
                    left = clean_span(trim_left(sentence[:match.start()]))
                    right = clean_span(trim_right(sentence[match.end():]))
                    cause, effect = (right, left) if orientation == "reverse" else (left, right)
                    if is_low_information_span(cause) or is_low_information_span(effect):
                        continue
                    quality = span_quality(cause, effect, cue_weight)
                    if quality < MIN_FRAME_QUALITY:
                        continue
                    candidates.append({
                        "corpus_type": corpus_type,
                        "doc_id": document["doc_id"],
                        "chunk_id": document["chunk_id"],
                        "sentence_id": document["sentence_id"],
                        "chunk_index": int(document["chunk_index"]),
                        "sentence_index": int(sentence_index),
                        "filename": document["filename"],
                        "source_file": document["source_file"],
                        "country": document["analysis_country"],
                        "heading_context": document.get("heading_context", ""),
                        "frame_sentence": normalize_source_text(sentence),
                        "context_window": document.get("context_window", ""),
                        "cause_span": cause,
                        "relation_type": relation_type,
                        "relation_cue": match.group(0),
                        "cue_orientation": orientation,
                        "effect_span": effect,
                        "structured_frame": f"{cause} [{relation_type}] {effect}",
                        "document_nmf_topic": int(document["topic"]),
                        "document_topic_confidence": float(
                            document.get("topic_confidence", np.nan)
                        ),
                        "frame_quality": quality,
                        "frame_signature": (
                            f"{normalise_span(cause)}|{relation_type}|"
                            f"{normalise_span(effect)}"
                        ),
                        "sentence_signature": normalise_span(sentence),
                        "extraction_method": "quality_scored_local_causal_frame",
                    })

    result = pd.DataFrame(candidates)
    if result.empty:
        return result

    result = result.sort_values("frame_quality", ascending=False)

    # Overlapping chunks often repeat the same sentence. Keep one frame per
    # document/signature and one strongest frame per relation family in the
    # same document sentence.
    result = result.drop_duplicates(
        subset=["corpus_type", "doc_id", "frame_signature"],
        keep="first",
    )
    result = result.drop_duplicates(
        subset=["corpus_type", "doc_id", "sentence_signature", "relation_type"],
        keep="first",
    )
    result = result.sort_values(
        ["doc_id", "chunk_index", "sentence_index", "relation_type"]
    ).reset_index(drop=True)
    result.insert(
        0,
        "frame_id",
        [f"{corpus_type}_frame_{index:05d}" for index in range(len(result))],
    )

    source_counts = result.groupby(
        ["corpus_type", "source_file"]
    )["frame_id"].transform("count")
    result["source_balanced_weight"] = 1.0 / source_counts.clip(lower=1)
    return result


policy_frames = extract_frames(policy_chunks, "policy")
sentiment_frames = extract_frames(sentiment_chunks, "sentiment")
synthetic_frames = extract_frames(synthetic_chunks, "synthetic_sentiment")

if policy_frames.empty or sentiment_frames.empty:
    raise ValueError("Independent frame extraction produced no empirical frames.")

pd.concat([policy_frames, sentiment_frames], ignore_index=True).to_csv(
    OUTPUT_DIR / "causal_frame_extraction_audit.csv",
    index=False,
)

extraction_summary = pd.concat(
    [
        policy_frames.assign(corpus="policy"),
        sentiment_frames.assign(corpus="sentiment"),
        synthetic_frames.assign(corpus="synthetic_sentiment"),
    ],
    ignore_index=True,
).groupby(["corpus", "relation_type"], as_index=False).agg(
    frames=("frame_id", "count"),
    documents=("doc_id", "nunique"),
    sources=("source_file", "nunique"),
    mean_frame_quality=("frame_quality", "mean"),
)
extraction_summary.to_csv(
    OUTPUT_DIR / "causal_frame_extraction_summary.csv",
    index=False,
)

frame_validation_rows = []
for corpus_name, extracted in [
    ("policy", policy_frames),
    ("sentiment", sentiment_frames),
    ("synthetic_sentiment", synthetic_frames),
]:
    residue_mask = extracted["frame_sentence"].fillna("").astype(str).str.contains(
        r"https?://|h\s+tt\s+ps|www\.|doi\.org|"
        r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}|\|",
        case=False,
        regex=True,
    )
    duplicate_rows = extracted.duplicated(
        ["corpus_type", "doc_id", "frame_signature"]
    )
    frame_validation_rows.append({
        "corpus": corpus_name,
        "frames": len(extracted),
        "minimum_frame_quality": float(extracted["frame_quality"].min())
        if not extracted.empty else np.nan,
        "mean_frame_quality": float(extracted["frame_quality"].mean())
        if not extracted.empty else np.nan,
        "source_residue_rows": int(residue_mask.sum()),
        "within_document_duplicate_rows": int(duplicate_rows.sum()),
        "maximum_sentence_words": int(
            extracted["frame_sentence"].fillna("").str.split().str.len().max()
        ) if not extracted.empty else 0,
    })

frame_validation = pd.DataFrame(frame_validation_rows)
frame_validation.to_csv(
    OUTPUT_DIR / "causal_frame_quality_validation.csv",
    index=False,
)
if (
    frame_validation["source_residue_rows"].gt(0).any()
    or frame_validation["within_document_duplicate_rows"].gt(0).any()
    or frame_validation["minimum_frame_quality"].lt(MIN_FRAME_QUALITY).any()
):
    raise ValueError(
        "Frame quality validation failed. See causal_frame_quality_validation.csv."
    )

print("Policy frames:", len(policy_frames))
print("Sentiment frames:", len(sentiment_frames))
print("Synthetic frames:", len(synthetic_frames))
print(extraction_summary.to_string(index=False))


Policy frames: 3196
Sentiment frames: 655
Synthetic frames: 674
             corpus          relation_type  frames  documents  sources  mean_frame_quality
             policy    causes_or_increases     915         49       49            0.731761
             policy    enables_or_supports    1447         50       50            0.698370
             policy    expected_to_improve     290         42       42            0.676113
             policy    reduces_or_prevents     179         39       39            0.720635
             policy requires_or_depends_on     251         40       40            0.680097
             policy     risks_or_threatens     114         31       31            0.721160
          sentiment    causes_or_increases     267         14       14            0.750898
          sentiment    enables_or_supports     239         15       15            0.709533
          sentiment    expected_to_improve      45         11       11            0.734263
          sentiment    red

In [4]:
# ==========================================
# Step 3: Project frame spans to policy topics with calibrated fallback
# ==========================================

MIN_SPAN_TOPIC_SIMILARITY = 0.02
MIN_TOPIC_MARGIN = 0.003
policy_topics["topic_description"] = build_topic_description(policy_topics)

fit_texts = (
    policy_topics["topic_description"].tolist()
    + policy_frames["cause_span"].tolist()
    + policy_frames["effect_span"].tolist()
    + sentiment_frames["cause_span"].tolist()
    + sentiment_frames["effect_span"].tolist()
)

word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    strip_accents="unicode",
    stop_words=sorted(CAUSAL_EMBEDDING_STOPWORDS),
    max_features=22000,
)
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    sublinear_tf=True,
    max_features=30000,
)
word_matrix = word_vectorizer.fit_transform(fit_texts)
char_matrix = char_vectorizer.fit_transform(fit_texts)
n_topics = len(policy_topics)


def split_matrices(matrix):
    topic_matrix = matrix[:n_topics]
    offset = n_topics
    p_cause = matrix[offset:offset + len(policy_frames)]
    offset += len(policy_frames)
    p_effect = matrix[offset:offset + len(policy_frames)]
    offset += len(policy_frames)
    s_cause = matrix[offset:offset + len(sentiment_frames)]
    offset += len(sentiment_frames)
    s_effect = matrix[offset:offset + len(sentiment_frames)]
    return topic_matrix, p_cause, p_effect, s_cause, s_effect


w_topic, w_pc, w_pe, w_sc, w_se = split_matrices(word_matrix)
c_topic, c_pc, c_pe, c_sc, c_se = split_matrices(char_matrix)


def hybrid_similarity(word_spans, char_spans):
    return (
        0.65 * cosine_similarity(word_spans, w_topic)
        + 0.35 * cosine_similarity(char_spans, c_topic)
    )


def assign_topics(
    frame: pd.DataFrame,
    cause_similarity: np.ndarray,
    effect_similarity: np.ndarray,
) -> pd.DataFrame:
    result = frame.copy().reset_index(drop=True)
    topic_ids = policy_topics["topic"].astype(int).to_numpy()

    for side, similarities in [
        ("cause", cause_similarity),
        ("effect", effect_similarity),
    ]:
        order = np.argsort(similarities, axis=1)[:, ::-1]
        best_position = order[:, 0]
        second_position = order[:, 1]
        best_score = similarities[np.arange(len(result)), best_position]
        second_score = similarities[np.arange(len(result)), second_position]
        margin = best_score - second_score
        projected_topic = topic_ids[best_position]
        reliable = (
            (best_score >= MIN_SPAN_TOPIC_SIMILARITY)
            & (margin >= MIN_TOPIC_MARGIN)
        )
        assigned_topic = np.where(
            reliable,
            projected_topic,
            result["document_nmf_topic"].astype(int),
        )
        score_confidence = np.clip(best_score / 0.15, 0.0, 1.0)
        margin_confidence = np.clip(margin / 0.05, 0.0, 1.0)
        projection_confidence = (
            0.70 * score_confidence + 0.30 * margin_confidence
        )
        fallback_confidence = 0.55 * pd.to_numeric(
            result["document_topic_confidence"],
            errors="coerce",
        ).fillna(0.5).to_numpy()
        assignment_confidence = np.where(
            reliable,
            projection_confidence,
            fallback_confidence,
        )

        result[f"{side}_topic"] = assigned_topic.astype(int)
        result[f"{side}_projected_topic"] = projected_topic.astype(int)
        result[f"{side}_topic_similarity"] = best_score
        result[f"{side}_topic_margin"] = margin
        result[f"{side}_assignment_method"] = np.where(
            reliable,
            "hybrid_span_projection",
            "document_nmf_fallback",
        )
        result[f"{side}_assignment_confidence"] = assignment_confidence
        result[f"{side}_topic_code"] = result[f"{side}_topic"].map(
            lambda value: f"P{int(value)}"
        )

    result["analysis_weight"] = (
        result["source_balanced_weight"]
        * result["frame_quality"].clip(lower=0.20)
        * np.sqrt(
            result["cause_assignment_confidence"].clip(lower=0.05)
            * result["effect_assignment_confidence"].clip(lower=0.05)
        )
    )
    result["topic_assignment_method"] = (
        "hybrid_word_char_projection_with_document_fallback"
    )
    return result


policy_frames = assign_topics(
    policy_frames,
    hybrid_similarity(w_pc, c_pc),
    hybrid_similarity(w_pe, c_pe),
)
sentiment_frames = assign_topics(
    sentiment_frames,
    hybrid_similarity(w_sc, c_sc),
    hybrid_similarity(w_se, c_se),
)

if not synthetic_frames.empty:
    syn_word_cause = word_vectorizer.transform(
        synthetic_frames["cause_span"].tolist()
    )
    syn_word_effect = word_vectorizer.transform(
        synthetic_frames["effect_span"].tolist()
    )
    syn_char_cause = char_vectorizer.transform(
        synthetic_frames["cause_span"].tolist()
    )
    syn_char_effect = char_vectorizer.transform(
        synthetic_frames["effect_span"].tolist()
    )
    synthetic_frames = assign_topics(
        synthetic_frames,
        hybrid_similarity(syn_word_cause, syn_char_cause),
        hybrid_similarity(syn_word_effect, syn_char_effect),
    )

policy_frames.to_csv(OUTPUT_DIR / "policy_causal_frames.csv", index=False)
sentiment_frames.to_csv(OUTPUT_DIR / "sentiment_causal_frames.csv", index=False)
synthetic_frames.to_csv(OUTPUT_DIR / "synthetic_causal_frames.csv", index=False)

assignment_rows = []
for corpus_name, frame in [
    ("policy", policy_frames),
    ("sentiment", sentiment_frames),
]:
    for side in ["cause", "effect"]:
        assignment_rows.append({
            "corpus": corpus_name,
            "side": side,
            "frames": len(frame),
            "fallback_share": float(
                frame[f"{side}_assignment_method"]
                .eq("document_nmf_fallback")
                .mean()
            ),
            "mean_similarity": float(frame[f"{side}_topic_similarity"].mean()),
            "zero_similarity_share": float(
                frame[f"{side}_topic_similarity"].le(1e-12).mean()
            ),
            "mean_margin": float(frame[f"{side}_topic_margin"].mean()),
            "mean_assignment_confidence": float(
                frame[f"{side}_assignment_confidence"].mean()
            ),
        })

assignment_diagnostics = pd.DataFrame(assignment_rows)
assignment_diagnostics.to_csv(
    OUTPUT_DIR / "topic_assignment_diagnostics.csv",
    index=False,
)
print(assignment_diagnostics.to_string(index=False))


/home/nsirim/Jupyter/bert_env/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['aout', 'ca', 'meme', 'tres'] not in stop_words.
  warnings.warn(


   corpus   side  frames  fallback_share  mean_similarity  zero_similarity_share  mean_margin  mean_assignment_confidence
   policy  cause    3196        0.318210         0.056833               0.001252     0.025003                    0.434309
   policy effect    3196        0.307259         0.060004               0.000313     0.026256                    0.447937
sentiment  cause     655        0.351145         0.049148               0.000000     0.018513                    0.409290
sentiment effect     655        0.334351         0.054150               0.000000     0.022709                    0.438520


In [5]:
# ==========================================
# Step 4: Build quality-weighted global causal-frame networks
# ==========================================

EPSILON = 1e-8
all_topic_ids = policy_topics["topic"].astype(int).tolist()
all_relations = sorted(
    set(policy_frames["relation_type"]).union(sentiment_frames["relation_type"])
)
index = pd.MultiIndex.from_product(
    [all_topic_ids, all_relations, all_topic_ids],
    names=["cause_topic", "relation_type", "effect_topic"],
)


def relation_distribution(
    frame: pd.DataFrame,
    weight_column: str = "analysis_weight",
) -> pd.Series:
    grouped = frame.groupby(
        ["cause_topic", "relation_type", "effect_topic"]
    )[weight_column].sum().reindex(index, fill_value=0.0)
    total = grouped.sum()
    return grouped / total if total > 0 else grouped


def edge_support(frame: pd.DataFrame, prefix: str) -> pd.DataFrame:
    return frame.groupby(
        ["cause_topic", "relation_type", "effect_topic"],
        as_index=False,
    ).agg(**{
        f"{prefix}_frames": ("frame_id", "count"),
        f"{prefix}_documents": ("doc_id", "nunique"),
        f"{prefix}_sources": ("source_file", "nunique"),
        f"{prefix}_mean_quality": ("frame_quality", "mean"),
        f"{prefix}_mean_assignment_confidence": (
            "cause_assignment_confidence",
            "mean",
        ),
    })


policy_distribution = relation_distribution(policy_frames)
sentiment_distribution = relation_distribution(sentiment_frames)

global_gaps = pd.DataFrame({
    "policy_share": policy_distribution,
    "sentiment_share": sentiment_distribution,
}).reset_index()
global_gaps["signed_gap"] = (
    global_gaps["policy_share"] - global_gaps["sentiment_share"]
)
global_gaps["absolute_gap"] = global_gaps["signed_gap"].abs()
global_gaps["log_ratio"] = np.log(
    (global_gaps["policy_share"] + EPSILON)
    / (global_gaps["sentiment_share"] + EPSILON)
)
global_gaps["cause_topic_code"] = global_gaps["cause_topic"].map(
    lambda value: f"P{int(value)}"
)
global_gaps["effect_topic_code"] = global_gaps["effect_topic"].map(
    lambda value: f"P{int(value)}"
)

global_gaps = global_gaps.merge(
    edge_support(policy_frames, "policy"),
    on=["cause_topic", "relation_type", "effect_topic"],
    how="left",
).merge(
    edge_support(sentiment_frames, "sentiment"),
    on=["cause_topic", "relation_type", "effect_topic"],
    how="left",
)
support_columns = [
    column
    for column in global_gaps.columns
    if column.endswith(("_frames", "_documents", "_sources"))
]
global_gaps[support_columns] = global_gaps[support_columns].fillna(0).astype(int)
global_gaps["documentary_support_score"] = np.log1p(
    global_gaps["policy_sources"] + global_gaps["sentiment_sources"]
)
global_gaps["evidence_rank_score"] = (
    global_gaps["absolute_gap"]
    * global_gaps["documentary_support_score"].clip(lower=0.25)
)
global_gaps.to_csv(
    OUTPUT_DIR / "global_causal_frame_network_gaps.csv",
    index=False,
)

js_distance = float(
    jensenshannon(
        policy_distribution.to_numpy(),
        sentiment_distribution.to_numpy(),
    )
)

relation_summary = global_gaps.groupby("relation_type", as_index=False).agg(
    policy_share=("policy_share", "sum"),
    sentiment_share=("sentiment_share", "sum"),
    total_absolute_gap=("absolute_gap", "sum"),
)
relation_summary["signed_gap"] = (
    relation_summary["policy_share"] - relation_summary["sentiment_share"]
)
relation_summary.to_csv(
    OUTPUT_DIR / "global_causal_frame_relation_summary.csv",
    index=False,
)

cause_contribution = global_gaps.groupby(
    "cause_topic",
    as_index=False,
)["absolute_gap"].sum().rename(
    columns={"absolute_gap": "cause_gap_contribution"}
)
effect_contribution = global_gaps.groupby(
    "effect_topic",
    as_index=False,
)["absolute_gap"].sum().rename(
    columns={
        "effect_topic": "cause_topic",
        "absolute_gap": "effect_gap_contribution",
    }
)
topic_gap = cause_contribution.merge(
    effect_contribution,
    on="cause_topic",
    how="outer",
).fillna(0.0).rename(columns={"cause_topic": "topic"})
topic_gap["relation_gap_magnitude"] = (
    topic_gap["cause_gap_contribution"]
    + topic_gap["effect_gap_contribution"]
)
topic_gap["topic_code"] = topic_gap["topic"].map(
    lambda value: f"P{int(value)}"
)

quality_rows = []
for topic in all_topic_ids:
    involved = pd.concat([
        policy_frames[
            policy_frames["cause_topic"].eq(topic)
            | policy_frames["effect_topic"].eq(topic)
        ],
        sentiment_frames[
            sentiment_frames["cause_topic"].eq(topic)
            | sentiment_frames["effect_topic"].eq(topic)
        ],
    ], ignore_index=True)
    quality_rows.append({
        "topic": topic,
        "frame_support": len(involved),
        "unique_documents": int(involved["doc_id"].nunique()),
        "unique_sources": int(involved["source_file"].nunique()),
        "mean_frame_quality": float(involved["frame_quality"].mean())
        if not involved.empty else 0.0,
        "mean_assignment_confidence": float(pd.concat([
            involved["cause_assignment_confidence"],
            involved["effect_assignment_confidence"],
        ]).mean()) if not involved.empty else 0.0,
        "fallback_share": float(pd.concat([
            involved["cause_assignment_method"].eq("document_nmf_fallback"),
            involved["effect_assignment_method"].eq("document_nmf_fallback"),
        ]).mean()) if not involved.empty else 1.0,
    })

quality_by_topic = pd.DataFrame(quality_rows)
topic_gap = topic_gap.merge(quality_by_topic, on="topic", how="left")
topic_gap.to_csv(
    OUTPUT_DIR / "global_causal_frame_gap_by_topic.csv",
    index=False,
)
quality_by_topic.to_csv(
    OUTPUT_DIR / "global_frame_topic_assignment_quality.csv",
    index=False,
)

robustness_rows = []
for specification, p_frame, s_frame, weight in [
    (
        "quality_weighted_primary",
        policy_frames,
        sentiment_frames,
        "analysis_weight",
    ),
    (
        "source_balanced_only",
        policy_frames,
        sentiment_frames,
        "source_balanced_weight",
    ),
    (
        "high_quality_only",
        policy_frames[policy_frames["frame_quality"].ge(0.60)],
        sentiment_frames[sentiment_frames["frame_quality"].ge(0.60)],
        "analysis_weight",
    ),
    (
        "projected_only",
        policy_frames[
            policy_frames["cause_assignment_method"].eq("hybrid_span_projection")
            & policy_frames["effect_assignment_method"].eq("hybrid_span_projection")
        ],
        sentiment_frames[
            sentiment_frames["cause_assignment_method"].eq("hybrid_span_projection")
            & sentiment_frames["effect_assignment_method"].eq("hybrid_span_projection")
        ],
        "source_balanced_weight",
    ),
]:
    if p_frame.empty or s_frame.empty:
        continue
    p_dist = relation_distribution(p_frame, weight)
    s_dist = relation_distribution(s_frame, weight)
    distance = float(
        jensenshannon(p_dist.to_numpy(), s_dist.to_numpy())
    )
    robustness_rows.append({
        "specification": specification,
        "policy_frames": len(p_frame),
        "sentiment_frames": len(s_frame),
        "jensen_shannon_divergence": distance ** 2,
    })

robustness = pd.DataFrame(robustness_rows)
robustness.to_csv(
    OUTPUT_DIR / "global_causal_frame_network_robustness.csv",
    index=False,
)


def source_edge_vectors(frame: pd.DataFrame) -> tuple[list[str], np.ndarray]:
    sources = sorted(frame["source_file"].astype(str).unique())
    vectors = []
    for source in sources:
        subset = frame[frame["source_file"].astype(str).eq(source)]
        grouped = subset.groupby(
            ["cause_topic", "relation_type", "effect_topic"]
        )["analysis_weight"].sum().reindex(index, fill_value=0.0)
        vector = grouped.to_numpy(dtype=float)
        vectors.append(vector)
    return sources, np.vstack(vectors)


policy_sources, policy_source_vectors = source_edge_vectors(policy_frames)
sentiment_sources, sentiment_source_vectors = source_edge_vectors(sentiment_frames)
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_values = []
for iteration in range(BOOTSTRAP_ITERATIONS):
    p_indices = rng.integers(0, len(policy_sources), len(policy_sources))
    s_indices = rng.integers(0, len(sentiment_sources), len(sentiment_sources))
    p_vector = policy_source_vectors[p_indices].sum(axis=0)
    s_vector = sentiment_source_vectors[s_indices].sum(axis=0)
    p_vector = p_vector / p_vector.sum()
    s_vector = s_vector / s_vector.sum()
    bootstrap_values.append({
        "iteration": iteration,
        "jensen_shannon_divergence": float(
            jensenshannon(p_vector, s_vector) ** 2
        ),
    })

bootstrap_iterations = pd.DataFrame(bootstrap_values)
bootstrap_iterations.to_csv(
    OUTPUT_DIR / "causal_frame_network_bootstrap_iterations.csv",
    index=False,
)
bootstrap_summary = pd.DataFrame([{
    "iterations": BOOTSTRAP_ITERATIONS,
    "bootstrap_unit": "source_file",
    "mean_jensen_shannon_divergence": float(
        bootstrap_iterations["jensen_shannon_divergence"].mean()
    ),
    "source_resampling_p025": float(
        bootstrap_iterations["jensen_shannon_divergence"].quantile(0.025)
    ),
    "source_resampling_p975": float(
        bootstrap_iterations["jensen_shannon_divergence"].quantile(0.975)
    ),
}])
bootstrap_summary.to_csv(
    OUTPUT_DIR / "causal_frame_network_bootstrap_summary.csv",
    index=False,
)

pd.DataFrame([{
    "jensen_shannon_distance": js_distance,
    "jensen_shannon_divergence": js_distance ** 2,
    "policy_frames": len(policy_frames),
    "sentiment_frames": len(sentiment_frames),
    "policy_sources": len(policy_sources),
    "sentiment_sources": len(sentiment_sources),
}]).to_csv(
    OUTPUT_DIR / "global_causal_frame_network_summary.csv",
    index=False,
)

print("Jensen-Shannon divergence:", round(js_distance ** 2, 6))
print(robustness.to_string(index=False))
print(bootstrap_summary.to_string(index=False))


Jensen-Shannon divergence: 0.25963
           specification  policy_frames  sentiment_frames  jensen_shannon_divergence
quality_weighted_primary           3196               655                   0.259630
    source_balanced_only           3196               655                   0.242488
       high_quality_only           2513               547                   0.273795
          projected_only           1540               294                   0.296440
 iterations bootstrap_unit  mean_jensen_shannon_divergence  source_resampling_p025  source_resampling_p975
        250    source_file                        0.335857                0.266272                0.415718


In [6]:
# ==========================================
# Step 5: Build country-level causal-frame network gaps
# ==========================================

country_status_rows = []
country_gap_frames = []
for country in sorted(
    set(policy_frames["country"]).union(sentiment_frames["country"])
):
    policy_subset = policy_frames[policy_frames["country"].eq(country)]
    sentiment_subset = sentiment_frames[sentiment_frames["country"].eq(country)]
    policy_sources = int(policy_subset["source_file"].nunique())
    sentiment_sources = int(sentiment_subset["source_file"].nunique())
    sufficient = (
        len(policy_subset) >= MIN_COUNTRY_ITEMS
        and len(sentiment_subset) >= MIN_COUNTRY_ITEMS
        and policy_sources >= MIN_COUNTRY_SOURCES
        and sentiment_sources >= MIN_COUNTRY_SOURCES
    )
    country_status_rows.append({
        "country": country,
        "policy_frames": len(policy_subset),
        "sentiment_frames": len(sentiment_subset),
        "policy_sources": policy_sources,
        "sentiment_sources": sentiment_sources,
        "coverage_status": "included" if sufficient else "insufficient evidence",
    })
    if not sufficient:
        continue

    p_distribution = relation_distribution(policy_subset)
    s_distribution = relation_distribution(sentiment_subset)
    frame = pd.DataFrame({
        "policy_share": p_distribution,
        "sentiment_share": s_distribution,
    }).reset_index()
    frame["signed_gap"] = frame["policy_share"] - frame["sentiment_share"]
    frame["absolute_gap"] = frame["signed_gap"].abs()
    frame["country"] = country
    country_gap_frames.append(frame)

country_status = pd.DataFrame(country_status_rows)
country_gaps = (
    pd.concat(country_gap_frames, ignore_index=True)
    if country_gap_frames
    else pd.DataFrame()
)
country_status.to_csv(
    OUTPUT_DIR / "country_causal_frame_status.csv",
    index=False,
)
country_gaps.to_csv(
    OUTPUT_DIR / "country_causal_frame_network_gaps.csv",
    index=False,
)

if not country_gaps.empty:
    cause_side = country_gaps.groupby(
        ["country", "cause_topic"],
        as_index=False,
    )["absolute_gap"].sum().rename(columns={
        "cause_topic": "topic",
        "absolute_gap": "cause_gap_contribution",
    })
    effect_side = country_gaps.groupby(
        ["country", "effect_topic"],
        as_index=False,
    )["absolute_gap"].sum().rename(columns={
        "effect_topic": "topic",
        "absolute_gap": "effect_gap_contribution",
    })
    country_topic_gap = cause_side.merge(
        effect_side,
        on=["country", "topic"],
        how="outer",
    ).fillna(0.0)
    country_topic_gap["relation_gap_magnitude"] = (
        country_topic_gap["cause_gap_contribution"]
        + country_topic_gap["effect_gap_contribution"]
    )
    country_topic_gap["topic_code"] = country_topic_gap["topic"].map(
        lambda value: f"P{int(value)}"
    )
else:
    country_topic_gap = pd.DataFrame()

country_topic_gap.to_csv(
    OUTPUT_DIR / "country_causal_frame_gap_by_topic.csv",
    index=False,
)
print(country_status.to_string(index=False))


  country  policy_frames  sentiment_frames  policy_sources  sentiment_sources       coverage_status
australia            640                 0              17                  0 insufficient evidence
   france            325                27               6                  3              included
  ireland            770               148              15                  3              included
    other            453               480               2                  9              included
      usa           1008                 0              15                  0 insufficient evidence


In [7]:
# ==========================================
# Step 6: Mine original-document evidence and source support
# ==========================================

# Rank edges by gap size and documentary breadth rather than gap size alone.
top_edges = global_gaps.sort_values(
    ["evidence_rank_score", "absolute_gap"],
    ascending=False,
).head(30).copy()

frame_pool = pd.concat(
    [policy_frames, sentiment_frames],
    ignore_index=True,
)
evidence_rows = []
for _, edge in top_edges.iterrows():
    matching = frame_pool[
        frame_pool["cause_topic"].eq(edge["cause_topic"])
        & frame_pool["relation_type"].eq(edge["relation_type"])
        & frame_pool["effect_topic"].eq(edge["effect_topic"])
    ].sort_values(
        ["analysis_weight", "frame_quality"],
        ascending=False,
    )

    for corpus_type in ["policy", "sentiment"]:
        candidate = (
            matching[matching["corpus_type"].eq(corpus_type)]
            .drop_duplicates("source_file")
            .head(2)
        )
        for _, row in candidate.iterrows():
            evidence_rows.append({
                "cause_topic": int(edge["cause_topic"]),
                "cause_topic_code": f"P{int(edge['cause_topic'])}",
                "relation_type": edge["relation_type"],
                "effect_topic": int(edge["effect_topic"]),
                "effect_topic_code": f"P{int(edge['effect_topic'])}",
                "signed_gap": float(edge["signed_gap"]),
                "absolute_gap": float(edge["absolute_gap"]),
                "evidence_rank_score": float(edge["evidence_rank_score"]),
                "policy_sources": int(edge["policy_sources"]),
                "sentiment_sources": int(edge["sentiment_sources"]),
                "evidence_corpus": corpus_type,
                "frame_id": row["frame_id"],
                "chunk_id": row["chunk_id"],
                "doc_id": row["doc_id"],
                "source_file": row["source_file"],
                "heading_context": row["heading_context"],
                "frame_sentence": row["frame_sentence"],
                "cause_span": row["cause_span"],
                "relation_cue": row["relation_cue"],
                "effect_span": row["effect_span"],
                "frame_quality": row["frame_quality"],
                "cause_assignment_method": row["cause_assignment_method"],
                "effect_assignment_method": row["effect_assignment_method"],
                "analysis_weight": row["analysis_weight"],
            })

frame_evidence = pd.DataFrame(evidence_rows)
frame_evidence.to_csv(
    OUTPUT_DIR / "global_causal_frame_gap_evidence.csv",
    index=False,
)

source_support = frame_pool.groupby(
    ["corpus_type", "cause_topic", "relation_type", "effect_topic"],
    as_index=False,
).agg(
    frames=("frame_id", "count"),
    unique_documents=("doc_id", "nunique"),
    unique_sources=("source_file", "nunique"),
    mean_frame_quality=("frame_quality", "mean"),
    mean_analysis_weight=("analysis_weight", "mean"),
)
source_support.to_csv(
    OUTPUT_DIR / "global_causal_frame_source_support.csv",
    index=False,
)
print(frame_evidence.head(12).to_string(index=False))


 cause_topic cause_topic_code          relation_type  effect_topic effect_topic_code  signed_gap  absolute_gap  evidence_rank_score  policy_sources  sentiment_sources evidence_corpus              frame_id                                                                                                                                      chunk_id                                                                                                                             doc_id                                                                                                                                          source_file                                                               heading_context                                                                                                                                                                                                                                                                                                        

In [8]:
# ==========================================
# Step 7: Run synthetic-data sensitivity independently
# ==========================================

synthetic_sensitivity = pd.DataFrame()
synthetic_network_summary = pd.DataFrame()

if not synthetic_frames.empty:
    synthetic_distribution = relation_distribution(synthetic_frames)
    synthetic_gaps = pd.DataFrame({
        "policy_share": policy_distribution,
        "empirical_sentiment_share": sentiment_distribution,
        "synthetic_sentiment_share": synthetic_distribution,
    }).reset_index()
    synthetic_gaps["empirical_absolute_gap"] = (
        synthetic_gaps["policy_share"]
        - synthetic_gaps["empirical_sentiment_share"]
    ).abs()
    synthetic_gaps["synthetic_absolute_gap"] = (
        synthetic_gaps["policy_share"]
        - synthetic_gaps["synthetic_sentiment_share"]
    ).abs()
    synthetic_gaps["absolute_gap_change"] = (
        synthetic_gaps["synthetic_absolute_gap"]
        - synthetic_gaps["empirical_absolute_gap"]
    )
    synthetic_sensitivity = synthetic_gaps.groupby(
        "relation_type",
        as_index=False,
    ).agg(
        empirical_absolute_gap=("empirical_absolute_gap", "sum"),
        synthetic_absolute_gap=("synthetic_absolute_gap", "sum"),
        absolute_gap_change=("absolute_gap_change", "sum"),
    )
    synthetic_sensitivity.to_csv(
        OUTPUT_DIR / "synthetic_causal_frame_sensitivity.csv",
        index=False,
    )

    empirical_js = float(
        jensenshannon(
            policy_distribution.to_numpy(),
            sentiment_distribution.to_numpy(),
        ) ** 2
    )
    synthetic_js = float(
        jensenshannon(
            policy_distribution.to_numpy(),
            synthetic_distribution.to_numpy(),
        ) ** 2
    )
    synthetic_network_summary = pd.DataFrame([{
        "empirical_jensen_shannon_divergence": empirical_js,
        "synthetic_jensen_shannon_divergence": synthetic_js,
        "divergence_change": synthetic_js - empirical_js,
        "synthetic_frames": len(synthetic_frames),
    }])
    synthetic_network_summary.to_csv(
        OUTPUT_DIR / "synthetic_causal_frame_network_summary.csv",
        index=False,
    )

print("Synthetic sensitivity rows:", len(synthetic_sensitivity))
if not synthetic_network_summary.empty:
    print(synthetic_network_summary.to_string(index=False))


Synthetic sensitivity rows: 6
 empirical_jensen_shannon_divergence  synthetic_jensen_shannon_divergence  divergence_change  synthetic_frames
                             0.25963                             0.361086           0.101455               674


In [9]:
# ==========================================
# Step 8: Generate figures with the same blue/orange conventions
# ==========================================


def aggregate_matrix(distribution: pd.Series) -> pd.DataFrame:
    return distribution.groupby(
        ["cause_topic", "effect_topic"]
    ).sum().unstack(fill_value=0.0)


policy_matrix = aggregate_matrix(policy_distribution)
sentiment_matrix = aggregate_matrix(sentiment_distribution)
gap_matrix = policy_matrix - sentiment_matrix
common_vmax = max(
    float(policy_matrix.to_numpy().max()),
    float(sentiment_matrix.to_numpy().max()),
)
gap_limit = float(np.abs(gap_matrix.to_numpy()).max())

matrix_specs = [
    (
        policy_matrix,
        "global_policy_causal_frame_matrix.png",
        "Policy relation share",
        "Blues",
        0.0,
        common_vmax,
    ),
    (
        sentiment_matrix,
        "global_sentiment_causal_frame_matrix.png",
        "Sentiment relation share",
        "Oranges",
        0.0,
        common_vmax,
    ),
    (
        gap_matrix,
        "global_causal_frame_gap_matrix.png",
        "Policy minus sentiment share",
        GAP_CMAP,
        -gap_limit,
        gap_limit,
    ),
]

for matrix, filename, label, cmap, vmin, vmax in matrix_specs:
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(
        matrix.to_numpy(),
        aspect="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_xticklabels([f"P{int(value)}" for value in matrix.columns])
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_yticklabels([f"P{int(value)}" for value in matrix.index])
    ax.set_xlabel("Effect topic ID")
    ax.set_ylabel("Cause topic ID")
    fig.colorbar(image, ax=ax, label=label)
    fig.tight_layout()
    save_verified_png(fig, IMG_DIR / filename)

# Side-by-side relation proportions: blue policy and orange sentiment.
relation_plot = relation_summary.sort_values("relation_type").reset_index(drop=True)
x_positions = np.arange(len(relation_plot))
bar_width = 0.38
fig, ax = plt.subplots(figsize=(12.5, 6.3))
ax.bar(
    x_positions - bar_width / 2,
    relation_plot["policy_share"],
    bar_width,
    color=BLUE,
    label="policy",
)
ax.bar(
    x_positions + bar_width / 2,
    relation_plot["sentiment_share"],
    bar_width,
    color=ORANGE,
    label="sentiment",
)
ax.set_xticks(x_positions)
ax.set_xticklabels(relation_plot["relation_type"], rotation=35, ha="right")
ax.set_xlabel("Causal relation type")
ax.set_ylabel("Source-balanced share")
ax.legend(loc="upper right")
fig.tight_layout()
save_verified_png(
    fig,
    IMG_DIR / "global_causal_frame_relation_types.png",
)

# Topic-level gap contributions retain cause/effect structure without implying
# a policy/sentiment topic crosswalk.
topic_plot = topic_gap.sort_values("topic").copy()
x_positions = np.arange(len(topic_plot))
fig, ax = plt.subplots(figsize=(11.5, 6.2))
ax.bar(
    x_positions - bar_width / 2,
    topic_plot["cause_gap_contribution"],
    bar_width,
    color=BLUE,
    label="cause-side contribution",
)
ax.bar(
    x_positions + bar_width / 2,
    topic_plot["effect_gap_contribution"],
    bar_width,
    color=ORANGE,
    label="effect-side contribution",
)
ax.set_xticks(x_positions)
ax.set_xticklabels(topic_plot["topic_code"])
ax.set_xlabel("Frozen policy topic ID")
ax.set_ylabel("Causal-frame gap contribution")
ax.legend(loc="upper right")
fig.tight_layout()
save_verified_png(
    fig,
    IMG_DIR / "global_causal_frame_gap_by_topic_id.png",
)

if not country_topic_gap.empty:
    matrix = country_topic_gap.pivot(
        index="topic_code",
        columns="country",
        values="relation_gap_magnitude",
    )
    fig, ax = plt.subplots(
        figsize=(7 + 1.3 * len(matrix.columns), 7)
    )
    image = ax.imshow(
        matrix.to_numpy(),
        aspect="auto",
        cmap="Blues",
    )
    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns)
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.set_ylabel("Frozen policy topic ID")
    fig.colorbar(
        image,
        ax=ax,
        label="Causal-frame gap magnitude",
    )
    fig.tight_layout()
    save_verified_png(
        fig,
        IMG_DIR / "country_causal_frame_gap_by_topic_id.png",
    )

if not synthetic_sensitivity.empty:
    colors = np.where(
        synthetic_sensitivity["absolute_gap_change"].ge(0.0),
        ORANGE,
        BLUE,
    )
    fig, ax = plt.subplots(figsize=(11.5, 5.5))
    ax.bar(
        synthetic_sensitivity["relation_type"],
        synthetic_sensitivity["absolute_gap_change"],
        color=colors,
    )
    ax.axhline(0.0, color="black", linewidth=1)
    ax.tick_params(axis="x", rotation=35)
    ax.set_ylabel("Synthetic minus empirical gap magnitude")
    fig.tight_layout()
    save_verified_png(
        fig,
        IMG_DIR / "synthetic_causal_frame_sensitivity.png",
    )

fig, ax = plt.subplots(figsize=(10.5, 5.8))
ax.hist(
    bootstrap_iterations["jensen_shannon_divergence"],
    bins=20,
    color=BLUE,
    alpha=0.75,
    edgecolor="white",
)
ax.axvline(
    js_distance ** 2,
    color=ORANGE,
    linewidth=2,
    label="Observed divergence",
)
ax.set_xlabel("Source-bootstrap Jensen–Shannon divergence")
ax.set_ylabel("Bootstrap iterations")
ax.legend(loc="upper right")
fig.tight_layout()
save_verified_png(
    fig,
    IMG_DIR / "causal_frame_network_bootstrap_histogram.png",
)

expected_pngs = [
    IMG_DIR / "global_policy_causal_frame_matrix.png",
    IMG_DIR / "global_sentiment_causal_frame_matrix.png",
    IMG_DIR / "global_causal_frame_gap_matrix.png",
    IMG_DIR / "global_causal_frame_relation_types.png",
    IMG_DIR / "global_causal_frame_gap_by_topic_id.png",
    IMG_DIR / "causal_frame_network_bootstrap_histogram.png",
]
if not country_topic_gap.empty:
    expected_pngs.append(
        IMG_DIR / "country_causal_frame_gap_by_topic_id.png"
    )
if not synthetic_sensitivity.empty:
    expected_pngs.append(
        IMG_DIR / "synthetic_causal_frame_sensitivity.png"
    )

png_rows = []
for png_path in expected_pngs:
    exists = png_path.exists()
    readable = False
    image_mode = ""
    width = 0
    height = 0
    size_bytes = png_path.stat().st_size if exists else 0
    if exists and size_bytes > 0:
        with Image.open(png_path) as image:
            image.load()
            readable = True
            image_mode = image.mode
            width, height = image.size
    png_rows.append({
        "filename": png_path.name,
        "exists": exists,
        "readable": readable,
        "mode": image_mode,
        "width": width,
        "height": height,
        "size_bytes": size_bytes,
    })

png_validation = pd.DataFrame(png_rows)
png_validation.to_csv(OUTPUT_DIR / "png_validation.csv", index=False)
if not (
    png_validation["readable"]
    & png_validation["mode"].eq("RGB")
).all():
    raise OSError("One or more generated PNG files failed validation.")
print(png_validation.to_string(index=False))


Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/frame_network/global_policy_causal_frame_matrix.png size=(1235, 1103)
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/frame_network/global_sentiment_causal_frame_matrix.png size=(1235, 1103)
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/frame_network/global_causal_frame_gap_matrix.png size=(1244, 1102)
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/frame_network/global_causal_frame_relation_types.png size=(1982, 984)
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/frame_network/global_causal_frame_gap_by_topic_id.png size=(1822, 975)
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/frame_network/country_causal_frame_gap_by_topic_id.png size=(1617, 1103)
Saved verified PNG: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/frame_network/synthetic_causal_frame_sensitivity.pn

In [10]:
# ==========================================
# Step 9: Export run summary and references
# ==========================================

run_summary = pd.DataFrame([{
    "method": "causal_frame_network_divergence",
    "shared_cleaning_version": CLEANING_VERSION,
    "shared_inventory_sha256": shared_cleaning_metadata["inventory_sha256"],
    "shared_inventory_rows": shared_cleaning_metadata["inventory_rows"],
    "independent_source": "chunks_all.csv",
    "policy_frames": len(policy_frames),
    "sentiment_frames": len(sentiment_frames),
    "synthetic_frames": len(synthetic_frames),
    "minimum_frame_quality": MIN_FRAME_QUALITY,
    "topic_assignment_method": (
        "hybrid_word_char_projection_with_document_fallback"
    ),
    "minimum_span_similarity": MIN_SPAN_TOPIC_SIMILARITY,
    "minimum_topic_margin": MIN_TOPIC_MARGIN,
    "reads_semantic_coverage_outputs": False,
    "jensen_shannon_divergence": js_distance ** 2,
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "source_resampling_p025": float(
        bootstrap_summary["source_resampling_p025"].iloc[0]
    ),
    "source_resampling_p975": float(
        bootstrap_summary["source_resampling_p975"].iloc[0]
    ),
}])
run_summary.to_csv(
    OUTPUT_DIR / "frame_network_run_summary.csv",
    index=False,
)

pd.DataFrame([
    {
        "method": "NMF thematic projection",
        "citation_key": "lee2001algorithms",
    },
    {
        "method": "Causal frame extraction",
        "citation_key": "ma2025causal;liu2025large;wan2025large",
    },
    {
        "method": "Jensen-Shannon divergence",
        "citation_key": "lin1991divergence",
    },
    {
        "method": "Source bootstrap",
        "citation_key": "efron1979bootstrap",
    },
    {
        "method": "Sensitivity analysis",
        "citation_key": "saltelli2008global",
    },
    {
        "method": "Structural causal modelling caution",
        "citation_key": "pearl2009causality",
    },
]).to_csv(
    OUTPUT_DIR / "method_references.csv",
    index=False,
)

print(run_summary.to_string(index=False))
print("Causal-frame network divergence completed independently.")


                         method shared_cleaning_version                                          shared_inventory_sha256  shared_inventory_rows independent_source  policy_frames  sentiment_frames  synthetic_frames  minimum_frame_quality                            topic_assignment_method  minimum_span_similarity  minimum_topic_margin  reads_semantic_coverage_outputs  jensen_shannon_divergence  bootstrap_iterations  source_resampling_p025  source_resampling_p975
causal_frame_network_divergence      shared-causal-text 59f25d414f8672b5fd0423a5c35ea1452e3d3d051a4d045d2c378c142c40f96a                  21591     chunks_all.csv           3196               655               674                    0.4 hybrid_word_char_projection_with_document_fallback                     0.02                 0.003                            False                    0.25963                   250                0.266272                0.415718
Causal-frame network divergence completed independently.
